# **Introduction to AI Workshop: Example 05**

In the previous example, we constructed a neural network to generate text. We started with a text file and trained our neural network from scratch to predict the next token in a sequence.

In this example, we will use a **pre-trained language model** called **GPT-2** to generate text.

Unlike the neural network we constructed in the previous example, GPT-2 has already been trained on a large amount of text. This means that it has already learned many patterns in language and can generate text without any additional training.

We will first use GPT-2 to generate some text. We will then fine-tune the model using the same text file we used in the previous example.

**Fine-tuning** means taking a model that has already been trained and continuing to train it using a new dataset.

Finally, we will generate text again and see how fine-tuning has affected the results.

### **Part 1: Loading the Text Data**

First, we import the libraries we will need. We will use `TensorFlow` and `Keras` to work with our neural network. We will use `KerasHub` to load the pre-trained GPT-2 model.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras
import keras_hub
import re
import matplotlib.pyplot as plt

Next, we upload a text file that we will use to train our neural network.

In [2]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

Saving TheRedHeadedLeague.txt to TheRedHeadedLeague.txt


Let's explore the text to get a better understanding of the data. Notice that this is the same text we used in the previous example.

In [3]:
print("Number of characters:", len(text))
print("\nFirst 500 characters:")
print(text[:500])

Number of characters: 49357

First 500 characters:
THE RED-HEADED LEAGUE

I had called upon my friend, Mr. Sherlock Holmes, one day in the
autumn of last year and found him in deep conversation with a
very stout, florid-faced, elderly gentleman with fiery red hair.
With an apology for my intrusion, I was about to withdraw when
Holmes pulled me abruptly into the room and closed the door
behind me.

"You could not possibly have come at a better time, my dear
Watson," he said cordially.

"I was afraid that you were engaged."

"So I am. Very much so


### **Part 2: Preparing the Text Data**

Before we can use our text to fine-tune GPT-2, we need to prepare it.

First, we **remove unnecessary whitespace**. Notice that we do not convert the text to lowercase as we did in the previous example. GPT-2 was trained using text that preserves uppercase and lowercase letters, so we will keep the original capitalization.

In [4]:
text = re.sub(r'\s+', ' ', text).strip()

Next, we divide the text into smaller pieces called **chunks**.

We will use chunks of 500 characters, with an overlap of 100 characters between consecutive chunks. The overlap helps preserve the context between chunks.

In [5]:
CHUNK_SIZE = 500
OVERLAP = 100
chunks = []
for i in range(0, len(text), CHUNK_SIZE - OVERLAP):
    chunk = text[i:i + CHUNK_SIZE]
    if len(chunk.strip()) > 50:
        chunks.append(chunk)

Let's explore the chunks. **How many chunks do we have? What does the first chunk look like?**


In [6]:
print("Number of text chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Number of text chunks: 123

First chunk:
THE RED-HEADED LEAGUE I had called upon my friend, Mr. Sherlock Holmes, one day in the autumn of last year and found him in deep conversation with a very stout, florid-faced, elderly gentleman with fiery red hair. With an apology for my intrusion, I was about to withdraw when Holmes pulled me abruptly into the room and closed the door behind me. "You could not possibly have come at a better time, my dear Watson," he said cordially. "I was afraid that you were engaged." "So I am. Very much so." "


Notice that now we are dividing the text into chunks based on the number of characters, rather than the number of tokens.


### **Part 3: Loading the Pre-Trained GPT-2 Model**

We are now ready to load our pre-trained language model. We will use **GPT-2**, which was developed by OpenAI.

Before loading the model, we need to specify the maximum sequence length. This determines the maximum number of tokens that the model will process at one time.


In [7]:
SEQUENCE_LENGTH = 128

Next, we load the GPT-2 preprocessor.

The **preprocessor** takes care of converting our text into tokens and preparing it for the neural network. Unlike our previous example, we do not need to manually create a vocabulary or convert the tokens into integers. GPT-2 already has its own tokenizer.




In [8]:
preprocessor = (
    keras_hub.models.GPT2CausalLMPreprocessor.from_preset(
        "gpt2_base_en",
        sequence_length=SEQUENCE_LENGTH
    )
)

Finally, we load the pre-trained GPT-2 model. Notice that we use `from_preset()` to load the model. This loads the GPT-2 architecture along with the weights learned during its original training.

In [9]:
model = keras_hub.models.GPT2CausalLM.from_preset(
    "gpt2_base_en",
    preprocessor=preprocessor
)

### **Part 4: Generating Text with GPT-2**

Before we fine-tune our model, let's see how well GPT-2 can generate text using its original pre-trained weights.

We provide the model with a prompt and ask it to generate additional text. **Try doing this multiple times! What do you think of the generated text?**

In [10]:
prompt = "I have"
generated_text = model.generate(prompt, max_length=100)
print(generated_text)

I have a few things to say about the game and its development.

I think it's a very good game. It's very good in the sense that it's a great game, but it's also a game that is a lot of fun, and it has a very interesting story.

I think you have to be able to make a game that is a lot fun, and you have to do it with a lot of patience. I don't know if that's the


Notice that GPT-2 can already generate reasonably coherent text, even though we haven't trained it using our text file. This is because the model has already learned patterns in language during its original training.

### **Part 5: Fine-Tuning GPT-2**

We will now fine-tune GPT-2 using our text file.

Remember that we divided our text into chunks in Part 3. We will use these chunks as our training data.

We also divide the data into batches. Each batch will contain two text chunks.

In [11]:
BATCH_SIZE = 2
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(chunks)
    .shuffle(min(len(chunks), 1000))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

Let's look at how many training examples we have.

In [12]:
print("Training examples:", len(chunks))
print("Batch size:", BATCH_SIZE)

Training examples: 123
Batch size: 2


Next, we compile our model by specifying the optimization algorithm and loss function.

We use the Adam optimization algorithm and sparse categorical crossentropy as our loss function.

Notice that we use a relatively small **learning rate**. This is important because GPT-2 has already been trained, and we want to make relatively small adjustments to its existing weights rather than completely changing what it has learned.

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=2e-5
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    weighted_metrics=["accuracy"]
)

We can now fine-tune our model using our text data. Notice that we are using the same `fit()` function that we used in the previous example.

However, instead of training a neural network from scratch, we are continuing to train a model that has already learned patterns in language.


In [14]:
history = model.fit(
    train_dataset,
    epochs=10
)

Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 105s 642ms/step - accuracy: 0.2904 - loss: 3.5137
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 101ms/step - accuracy: 0.3323 - loss: 3.1450
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 103ms/step - accuracy: 0.3606 - loss: 2.8895
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 102ms/step - accuracy: 0.3861 - loss: 2.6818
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 104ms/step - accuracy: 0.4107 - loss: 2.4924
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 113ms/step - accuracy: 0.4433 - loss: 2.3160
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 105ms/step - accuracy: 0.4710 - loss: 2.1566
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.4997 - loss: 1.9996
Epoch 9/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 115ms/step - accuracy: 0.5250 - loss: 1.8643
Epoch 10/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.5560 - loss: 1.7210


### **Part 5: Generating Text After Fine-Tuning**

Now that we have fine-tuned GPT-2, let's see how well it generates text. We will use the same prompt that we used before fine-tuning.

**How does the generated text compare to the text generated before fine-tuning? Does it more closely resemble the writing style of our text source?**

In [15]:
prompt = "I have never"
generated_text = model.generate(prompt, max_length=100)
print(generated_text)

I have never heard of any such thing as the Freemasonry. From what I understand, it is a small but effective body which, when it gets started, will be ready to give the programme a serious look. I know very well that it would suit all the different trades and occupations of the man, but I would like only to suggest one of his most daring and daring occupations which he would not hesitate to apply to any other subject." "What do you call daring?" asked Holmes. "


We can also experiment with different prompts to see how they affect the text generated by our model. **Try changing the prompts and see what happens!**

### **Conclusion**

In this example, we used a pre-trained language model called GPT-2 to generate text.

Unlike the previous example, where we constructed and trained a neural network from scratch, we started with a model that had already been trained on a large amount of text.

We then fine-tuned the model using our own text data to adapt it to a particular writing style.

This illustrates one of the major advantages of pre-trained language models: instead of training a large neural network from scratch, we can start with a model that has already learned a great deal about language and adapt it to a new task or dataset.